In [79]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

In [80]:
pokemon = pd.read_csv("pokemon/data/raw_data/pokemon.csv")
combats = pd.read_csv("pokemon/data/raw_data/combats.csv")
pokemonfullstats = pd.read_csv("pokemon/data/raw_data/pokemonfullstats.csv")

### Xử lý cơ bản để merge tạo bộ dữ liệu cuối cùng, phần xử lý đã được đề cập trong các file trước

In [81]:
pokemon.loc[62, "Name"] = 'Primeape'
pokemonfullstats.drop(columns=['DexNumber'], inplace=True)
pokemon.drop(columns=['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary'], inplace=True)

In [82]:
merged_pokemon = pd.merge(pokemon, pokemonfullstats, left_on='Name', right_on='Name', how='outer')
merged_pokemon = merged_pokemon.dropna(subset=['#'])
merged_pokemon.sort_values(by=['#'], inplace=True, ignore_index=True)
merged_pokemon.rename(columns={'#': 'ID'}, inplace=True)

In [83]:
dirty_data = pd.merge(combats, merged_pokemon, left_on='First_pokemon', right_on='ID', how='inner')
dirty_data.drop(columns=['ID'], inplace=True)
dirty_data.rename(columns={c: f"{c}_P1" for c in dirty_data.columns[3:]}, inplace=True)

dirty_data = pd.merge(dirty_data, merged_pokemon, left_on='Second_pokemon', right_on='ID', how='inner')
dirty_data.drop(columns=['ID'], inplace=True)
dirty_data.rename(columns={c: f"{c}_P2" for c in dirty_data.columns[3 + merged_pokemon.shape[1] - 1:]}, inplace=True)

df = dirty_data.copy()

In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 99 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   First_pokemon          50000 non-null  int64  
 1   Second_pokemon         50000 non-null  int64  
 2   Winner                 50000 non-null  int64  
 3   Name_P1                50000 non-null  object 
 4   Type 1_P1              50000 non-null  object 
 5   Type 2_P1              25969 non-null  object 
 6   Type_P1                45020 non-null  object 
 7   Abilities_P1           45020 non-null  object 
 8   HiddenAbility_P1       45020 non-null  object 
 9   Generation_P1          45020 non-null  object 
 10  Hp_P1                  45020 non-null  float64
 11  Attack_P1              45020 non-null  float64
 12  Defense_P1             45020 non-null  float64
 13  SpecialAttack_P1       45020 non-null  float64
 14  SpecialDefense_P1      45020 non-null  float64
 15  Sp

Các công việc sẽ thực hiện trên data raw:
- Các giá trị NaN thay bằng -9999 thay vì 0 hay giá trị trung bình, để mô hình hiểu đây là một loại dữ liệu đặc biệt (dữ liệu rỗng)
- Tạo biến mục tiêu từ Winner
- Xoá bỏ các cột có dữ liệu dạng Object

In [85]:
df = df.fillna(-9999)

In [86]:
df['Winner'] = np.where(df['Winner'] == df['First_pokemon'], 0, 1)
df.drop(columns=['First_pokemon', 'Second_pokemon'], inplace = True)

In [87]:
obj_cols = df.select_dtypes(include=['object']).columns
obj_cols
df = df.drop(columns=obj_cols)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 73 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Winner                 50000 non-null  int32  
 1   Hp_P1                  50000 non-null  float64
 2   Attack_P1              50000 non-null  float64
 3   Defense_P1             50000 non-null  float64
 4   SpecialAttack_P1       50000 non-null  float64
 5   SpecialDefense_P1      50000 non-null  float64
 6   Speed_P1               50000 non-null  float64
 7   TotalStats_P1          50000 non-null  float64
 8   Weight_P1              50000 non-null  float64
 9   Height_P1              50000 non-null  float64
 10  CatchRate_P1           50000 non-null  float64
 11  EggCycles_P1           50000 non-null  float64
 12  BaseFriendship_P1      50000 non-null  float64
 13  IsLegendary_P1         50000 non-null  float64
 14  IsMythical_P1          50000 non-null  float64
 15  Is

In [88]:
X = df.drop(columns=['Winner'])
y = df['Winner']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

In [89]:
selector = RFE(estimator=LogisticRegression(solver='liblinear'), n_features_to_select=25, step=1)
selector.fit(X_train, y_train)

selected_cols = X_train.columns[selector.support_]
print(f"25 biến được chọn: {selected_cols}")

25 biến được chọn: Index(['Speed_P1', 'IsLegendary_P1', 'HasMega_P1', 'TotalEvoStages_P1',
       'DamageFromFighting_P1', 'DamageFromFlying_P1', 'DamageFromGhost_P1',
       'DamageFromSteel_P1', 'DamageFromWater_P1', 'DamageFromPsychic_P1',
       'DamageFromIce_P1', 'DamageFromDragon_P1', 'DamageFromDark_P1',
       'Speed_P2', 'IsUltraBeast_P2', 'TotalEvoStages_P2',
       'DamageFromNormal_P2', 'DamageFromFighting_P2', 'DamageFromFlying_P2',
       'DamageFromPoison_P2', 'DamageFromBug_P2', 'DamageFromGhost_P2',
       'DamageFromFire_P2', 'DamageFromPsychic_P2', 'DamageFromDark_P2'],
      dtype='object')


In [90]:
model = LogisticRegression(C = 0.00001)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

train = accuracy_score(y_train, y_train_pred)
test = accuracy_score(y_test, y_pred)

print(f"Train Accuracy: {train * 100:.2f}%")
print(f"Test Accuracy: {test * 100:.2f}%")

Train Accuracy: 83.32%
Test Accuracy: 83.04%


c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
